# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR² dataset package](https://doi.org/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://pypi.org/project/mlcroissant) library, referencing all dataset elements by their `@id` values for transparency and reproducibility.

### Dataset Source

The dataset Croissant schema is published at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

We use `mlcroissant` to load the dataset metadata and inspect the core details supplied by the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show main dataset description
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview

Let's review the available record sets and their associated fields, listing all entities by their `@id` for clarity and future referencing.

In [ ]:
# List all available RecordSets and their `@id`/field names
from mlcroissant.structures import RecordSet

print("Record Sets available in the dataset:")
record_sets = []
for obj in dataset.metadata.iter_objects():
    if isinstance(obj, RecordSet):
        record_sets.append(obj)

for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else '(Unnamed)'}")
    if rs.fields:
        print("  Fields:")
        for f in rs.fields:
            print(f"    - Field @id: {f.id}, name: {f.name if hasattr(f, 'name') else '(Unnamed)'}")
    else:
        print("  No fields listed.")
    print()
# Save list of record set @ids for later use
record_set_ids = [rs.id for rs in record_sets]
# If no record set is defined, fallback by inspecting possible default data tab
if not record_set_ids:
    # Use the top-level recordSet if present
    if hasattr(metadata, 'recordSet') and metadata.recordSet:
        record_set_ids = [metadata.recordSet] if isinstance(metadata.recordSet, str) else list(metadata.recordSet)

> **Tip**: You can use these `@id` values below to extract data from specific record sets and their fields.

## Show a few example records

Here is a preview of the first few records from each record set using its `@id`:

In [ ]:
for rs_id in record_set_ids:
    print(f"\nSample from RecordSet @id: {rs_id}")
    rec_iter = dataset.records(record_set=rs_id)
    for i, rec in enumerate(rec_iter):
        print(rec)
        if i == 2:
            break

## 3. Data Extraction

Now, let's load the tabular data from the record set(s) into pandas DataFrames using their `@id`. Data is always referenced and loaded by the record set's `@id`.

If there are multiple record sets, we load each to a separate DataFrame.

In [ ]:
dataframes = {}

# Load data from each RecordSet by @id
for rs_id in record_set_ids:
    recs = list(dataset.records(record_set=rs_id))
    if len(recs) > 0:
        dataframes[rs_id] = pd.DataFrame(recs)
        print(f"Loaded {len(dataframes[rs_id])} records for RecordSet: {rs_id}")
    else:
        print(f"No records loaded for {rs_id}.")
# For demonstration, select the first loaded DataFrame
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns for RecordSet {main_rs_id}:\n{dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())
else:
    main_rs_id = None

## 4. Exploratory Data Analysis (EDA)

We'll now perform several example data processing steps:
- Filtering records by a field (`@id`),
- Normalizing a numeric field (`@id`),
- Optionally grouping by a categorical field (`@id`).

All field references below are by their Croissant schema `@id` value.

In [ ]:
# Adjust these as appropriate for your schema:
# Replace with known field @id's and ensure the field is present (use print(dataframes[main_rs_id].columns))

from numpy import number

if main_rs_id:
    df = dataframes[main_rs_id]
    # Print all columns to identify candidate numeric fields
    print("Available fields (as column @id):", df.columns.tolist())
    # Try to automatically pick a numeric field (e.g. 'Age' or similar), else fallback to numeric columns
    candidate_numeric_ids = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    numeric_field_id = candidate_numeric_ids[0] if candidate_numeric_ids else None

    # We'll arbitrarily use threshold = 10 for demonstration (adjust as appropriate)
    threshold = 10

    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"First normalized values for {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, norm_col]].head())
    else:
        print("No numeric fields found for normalization/filtering.")

    # Attempt grouping by a likely categorical field
    # Pick a non-numeric field that isn't the index
    candidate_group_fields = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
    group_field_id = candidate_group_fields[0] if candidate_group_fields else None
    if group_field_id and numeric_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped (mean) by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable non-numeric field for grouping.")
else:
    print("No main record set DataFrame loaded for analysis.")

## 5. Visualization

Let's visualize the data distribution for the selected numeric field, grouped by a categorical field if one was found.

*Feel free to adjust plot parameters to your schema and intended analysis.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()
else:
    print("Insufficient data for plotting.")

## 6. Conclusion

- Using the Croissant schema and `mlcroissant` library, we've loaded, explored, and visualized clinical data for cancer survivors with secondary colorectal cancer, referencing all entities (record sets, fields, columns) by their `@id`.
- Such a reproducible, schema-driven approach facilitates transparent data use and enables easy downstream analysis across different tools and settings.

For further analysis:
- Investigate associations between clinical variables and MSI-H status,
- Explore more detailed categorizations or predictive modeling,
- Leverage other fields using their `@id` for complex analyses.

> **Learn more:** [mlcroissant documentation](https://github.com/mlcommons/croissant).
